# Week 19 - Monte Carlo Simulations

Monte Carlo simulation (MCS) is a computational technique widely used within the wind energy sector. They work by carrying out repeated random sampling, generally drawing from statistical distributions, to estimate uncertain outcomes. Their probabilistic nature enables them to be used for sensitivity analysis, confidence interval construction and risk analysis as well as point estimation. In wind energy, MCS are widely used to:
- Model variability in wind speeds
- Estimate long-term energy production (AEP)
- Estimate failure rate and associated costs
- Quantify uncertainty
- Support financial and risk analyses


We will walk through a complete MCS workflow: modeling wind speed, sampling, applying a turbine power curve, computing annual energy production, and analyzing uncertainty. First we will view a simpler example, estimating mean wind speed on a site. Wind speeds at most sites follow a Weibull distribution defined by shape parameter $k$ and scale parameter $\lambda$.

Classically, we view MCS as for loops with a random sample created at each iteration as below.

In [20]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

total_wind_speed = 0
for i in range(1000):
    lam = 7.5
    k = 2
    wind_speed = stats.weibull_min.rvs(k, lam)
    total_wind_speed += wind_speed

mean_wind_speed = total_wind_speed/1000
print(f'Estimated mean wind speed = {np.round(mean_wind_speed,2)} m/s')

Estimated mean wind speed = 8.4 m/s


However, in a case like this it is more sensible to instead generate a large random sample.

In [14]:
wind_speeds = stats.weibull_min.rvs(k, lam, size=1000)

We may now define a simplified turbine power curve with cut‑in at 3 m/s, rated power at 12 m/s, and cut‑out at 25 m/s, and use our random sample of wind_speeds to estimate random power output. It is then straightforward to calculate summary statistics for our estimated sample.

In [23]:
def power_curve(v):
    if v < 3:
        return 0
    elif v < 12:
        return (v - 3) / 9 * 3000
    elif v < 25:
        return 3000
    else:
        return 0


power_output = np.array([power_curve(v) for v in wind_speeds])

print(f'Mean power = {np.round(np.mean(power_output),2)} kW')
print(f'Median power = {np.round(np.median(power_output),2)} kW')
print(f'Capacity Factor = {np.round(100 * np.mean(power_output) / 3000, 2)}%')

Mean power = 1799.03 kW
Median power = 1781.39 kW
Capacity Factor = 59.97%


The mean power output may then be used to estimate annual energy production (AEP).

In [34]:
hours = 8760 # hours in a year
aep_kwh = np.mean(power_output) * hours
print(f'Estimated AEP {np.round(aep_kwh / 1000,2)} MWh')

Estimated AEP 15759.52 MWh


We can then further use MCS to understand uncertainty and construct a confidence interval.

In [33]:
runs = 200
results = np.zeros(runs)
for i in range(runs):
    w = wind_speeds = stats.weibull_min.rvs(k, lam, size=1000)
    p = np.array([power_curve(v) for v in w])
    aep = np.mean(p) * hours
    results[i] = aep
    results = np.array(results)

# Construct CI
print(np.percentile(results, [5, 95]) / 1000)

[15670.3893136  15809.98095962]


The ability to predict a confidence range allows us to quantify the risk of e.g. underproducing energy. In this example, we see we can say with around 95% certainty that this turbine will produce at least 15670 MWh this year. 
Of course, for a real site we have many more factors to consider - modelling downtime, wakes, curtailment, individual turbine conditions and effects on each-other and much more. Each of these requires its own probability function many with complex probabilistic sub-systems.

Let's for example just consider downtime. For each turbine we have to consider:
- Probability of each type of potential failure
- For each type of failure, consider the time until repair

Time until repair then requires modelling of all systems which affect this time:
- Time until detection
- Time until team available
- Time until weather is suitable for repairs
- Supply chain if a new part is required

This problem turns into a very complicated system with many moving parts. Monte Carlo Simulations allow us to simulate all of these probabilistic events in a broken down way, but they require us to have a good understanding of these probabilities. This is a major drawback of this method. Additionally, as this system gets more complex running it many times becomes more computationally complex and intensive. Running these simulations with the level of complexity that enables the accuracy we require in the wind energy sector becomes expensive, and sometimes even infeasbile. For this reason machine learning methods and analytical approaches are often more suitable. In the machine learning course next year, we will see lots of comparison with Monte Carlo Simulations as our 'baseline' and see when and where it it suitable, and when it is outperformed.